# Capè brand font — few-shot generation + digitisation

Run on **Google Colab** with **Runtime → Change runtime type → GPU**.

Two halves:
- **Part A–C** generate the missing/full Latin glyphs in the Capè brush style (GPU).
- **Part D** is deterministic: it vectorises a folder of glyph PNGs (from *any* source — the AI step, or letters Riccardo draws) and assembles an installable `cape.otf`. Part D works even without Part C.

Upload the `glyphs/` folder from the repo when prompted (the style references).

## Part A — environment

In [ ]:
import torch
assert torch.cuda.is_available(), 'Set Runtime -> GPU'
print('GPU:', torch.cuda.get_device_name(0))
import os, glob, zipfile, urllib.request
os.makedirs('/content/refs', exist_ok=True)
# Try to auto-fetch the glyph refs; if not published, fall back to manual upload of glyphs.zip
URL = 'https://raw.githubusercontent.com/w67jbrbvhv-byte/tastehub-assets/main/cape/font/glyphs.zip'
try:
    urllib.request.urlretrieve(URL, '/content/glyphs.zip')
except Exception:
    from google.colab import files
    print('Upload glyphs.zip (from the repo: ip-trademark/cape-gavi/font/glyphs.zip):')
    up = files.upload()
    fn = next(iter(up))
    os.replace(fn, '/content/glyphs.zip')
zipfile.ZipFile('/content/glyphs.zip').extractall('/content/refs')
print('style refs:', len(glob.glob('/content/refs/**/*.png', recursive=True)))

## Part B — few-shot font model (FontDiffuser)
Clones the model and fetches pretrained weights. Check the repo README for the current
inference entrypoint — the cell below follows the published `sample.py` interface; adjust
flag names if the repo has moved on.

In [ ]:
!git clone https://github.com/yeungchenwa/FontDiffuser.git
%cd FontDiffuser
!pip -q install -r requirements.txt accelerate diffusers transformers
# Pretrained checkpoint (see repo README 'ckpt' instructions; mirror may change):
!python -c "from huggingface_hub import snapshot_download; snapshot_download('yeungchenwa/FontDiffuser', local_dir='ckpt')" || echo 'If this fails, download ckpt per the repo README.'

## Part C — generate the alphabet in Capè style
For each target letter we pass a **content** image (the letter shape, from any plain bold font)
and **style** images (our Capè refs). Priority: the 8 missing caps **B F H J K Q S W X Y** —
but generate the full set so the face is internally consistent.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import os, glob
os.makedirs('content', exist_ok=True); os.makedirs('/content/out', exist_ok=True)
TARGETS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ.'
fnt = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 320)
for ch in TARGETS:
    im = Image.new('RGB', (512,512), 'white'); d = ImageDraw.Draw(im)
    name = 'dot' if ch=='.' else ch
    bb = d.textbbox((0,0), ch, font=fnt); w,h = bb[2]-bb[0], bb[3]-bb[1]
    d.text(((512-w)/2-bb[0],(512-h)/2-bb[1]), ch, font=fnt, fill='black')
    im.save(f'content/{name}.png')
styles = sorted(glob.glob('/content/refs/**/*.png', recursive=True))[:6]
print('content glyphs:', len(os.listdir('content')), '| style refs used:', len(styles))
# --- Inference (FontDiffuser sample interface; one glyph shown, then loop) ---
for ch in TARGETS:
    name = 'dot' if ch=='.' else ch
    cmd = (f"python sample.py --content_image_path content/{name}.png "
           f"--style_image_path {styles[0]} --save_image --save_image_dir /content/out "
           f"--ckpt_dir ckpt --device cuda:0")
    print(cmd)
    os.system(cmd)
print('generated:', len(glob.glob('/content/out/*.png')))

## Part D — digitise glyph PNGs → installable `cape.otf`  (deterministic)
Works on **any** folder of black-on-white glyph PNGs named by letter (e.g. `S.png`, `H.png`,
`dot.png`). Use the Part C output, the harvested refs, or letters Riccardo draws — same pipeline.

In [ ]:
!apt-get -qq install -y potrace fontforge >/dev/null
GLYPH_DIR = '/content/out'   # <-- folder of <Letter>.png black-on-white glyphs
import glob, os, subprocess
os.makedirs('/content/svg', exist_ok=True)
for p in glob.glob(f'{GLYPH_DIR}/*.png'):
    stem = os.path.splitext(os.path.basename(p))[0]
    subprocess.run(f"convert {p} -threshold 60% -bordercolor white -border 10 /content/svg/{stem}.pbm", shell=True)
    subprocess.run(f"potrace -s -o /content/svg/{stem}.svg /content/svg/{stem}.pbm", shell=True)
print('svgs:', len(glob.glob('/content/svg/*.svg')))

In [ ]:
ff = '''
import fontforge, glob, os
font = fontforge.font()
font.familyname = 'Cape'; font.fontname = 'Cape-Regular'; font.fullname = 'Cape Regular'
font.em = 1000
name2uni = {'dot': ord('.')}
for svg in glob.glob('/content/svg/*.svg'):
    stem = os.path.splitext(os.path.basename(svg))[0]
    cp = name2uni.get(stem, ord(stem[0]) if len(stem)==1 else None)
    if cp is None: continue
    g = font.createChar(cp)
    g.importOutlines(svg)
    g.width = 700
    g.autoHint()
font.generate('/content/cape.otf')
print('glyphs in font:', len([g for g in font.glyphs()]))
'''
open('build_font.py','w').write(ff)
!fontforge -script build_font.py
from google.colab import files
files.download('/content/cape.otf')

## After download
Send me `cape.otf` + the generated glyph PNGs. I'll review the S/H quality, fix spacing/kerning,
and finalise. Then 'OUTSMART THE DEVIL.' (and every future label) can be typeset in the real Capè hand.